# Task 4: Context-Aware Chatbot Using LangChain + RAG

## 📌 Objective
Build a conversational chatbot that:
- **Remembers conversation history** (context memory across turns)
- **Retrieves external knowledge** from a vectorized document store (Retrieval-Augmented Generation)
- **Answers grounded questions** by combining retrieved document chunks with chat history
- **Is deployable** as an interactive web app using Streamlit

## 🛠️ Tools & Technologies
| Component | Tool Used |
|---|---|
| Framework | LangChain |
| Embeddings | HuggingFace Sentence Transformers (`all-MiniLM-L6-v2`) |
| Vector Store | FAISS (Facebook AI Similarity Search) |
| LLM (generation) | HuggingFace `google/flan-t5-base` (local, free) — swappable with OpenAI/Anthropic API |
| Memory | LangChain `ConversationBufferMemory` |
| Data Source | Wikipedia articles (via `langchain_community.document_loaders.WikipediaLoader`) |
| Deployment | Streamlit |
| Language | Python 3.10+ |

## 🧠 Skills Gained
- Conversational AI development
- Document embedding & vector search
- Retrieval-Augmented Generation (RAG) pipeline design
- LLM integration and deployment

---
**Note on the LLM:** This notebook uses a small, free, local Hugging Face model (`flan-t5-base`) so the whole pipeline runs **without any paid API key**. If you have access to OpenAI or Anthropic API keys, a commented-out cell later shows how to swap in `ChatOpenAI` or `ChatAnthropic` for noticeably better answer quality.


## 📂 Dataset

**Custom corpus:** We use a small set of **Wikipedia pages** as our knowledge base (as suggested in the task instructions: *"Wikipedia pages, internal documents, or any knowledge base"*).

- Source: [Wikipedia](https://www.wikipedia.org/) (fetched live via the `wikipedia` Python package / `WikipediaLoader`)
- Example pages used below:
  - https://en.wikipedia.org/wiki/Artificial_intelligence
  - https://en.wikipedia.org/wiki/Machine_learning
  - https://en.wikipedia.org/wiki/Cybersecurity

> 💡 **You can replace this with your own documents** (PDFs, `.txt`, `.docx`, internal notes, FYP reports, etc.) by swapping `WikipediaLoader` for `PyPDFLoader`, `TextLoader`, or `Docx2txtLoader` — the rest of the pipeline (chunking → embedding → retrieval → memory) stays exactly the same.


## Step 1 — Install Dependencies
**Purpose:** Install LangChain, embedding, vector store, and Streamlit libraries needed for the whole pipeline.

In [1]:
!pip install -q langchain langchain-classic langchain-community langchain-huggingface langchain-text-splitters \
    sentence-transformers faiss-cpu wikipedia transformers accelerate streamlit


## Step 2 — Import Libraries
**Purpose:** Bring in all the LangChain modules, embedding tools, and HuggingFace components used across the notebook.

In [2]:
import warnings
warnings.filterwarnings("ignore")

from langchain_community.document_loaders import WikipediaLoader

# RecursiveCharacterTextSplitter moved packages across LangChain versions
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# In LangChain 1.x, memory & legacy chains moved to the separate
# "langchain-classic" package. We try the new location first,
# then fall back for older installs.
try:
    from langchain_classic.memory import ConversationBufferMemory
    from langchain_classic.chains import ConversationalRetrievalChain
except ImportError:
    from langchain.memory import ConversationBufferMemory
    from langchain.chains import ConversationalRetrievalChain

from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline as hf_pipeline

print("Libraries imported successfully")

Libraries imported successfully


## Step 3 — Load the Custom Corpus (Dataset)
**Purpose:** Fetch a small knowledge base of documents (Wikipedia pages) that our chatbot will be able to search through. Each topic becomes one or more LangChain `Document` objects with page content + metadata (source, title).

Dataset link(s): https://en.wikipedia.org/wiki/Artificial_intelligence , https://en.wikipedia.org/wiki/Machine_learning , https://en.wikipedia.org/wiki/Cybersecurity

In [3]:
import requests
from langchain_core.documents import Document

def fetch_wikipedia_document(title: str) -> Document:
    """Fetch the full plain-text extract of a Wikipedia page directly via
    the MediaWiki API (bypasses the unmaintained `wikipedia` pip package,
    which Wikipedia now blocks due to missing/invalid User-Agent headers)."""
    headers = {
        "User-Agent": "Task4-RAG-Chatbot/1.0 (educational project; contact: student@example.com)"
    }
    params = {
        "action": "query",
        "prop": "extracts",
        "explaintext": True,
        "titles": title,
        "format": "json",
        "redirects": 1,
    }
    response = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params=params,
        headers=headers,
        timeout=15,
    )
    response.raise_for_status()
    data = response.json()

    pages = data["query"]["pages"]
    page = next(iter(pages.values()))
    content = page.get("extract", "")
    page_title = page.get("title", title)

    if not content:
        raise ValueError(f"No content returned for '{title}' — page may not exist.")

    return Document(
        page_content=content,
        metadata={
            "title": page_title,
            "source": f"https://en.wikipedia.org/wiki/{page_title.replace(' ', '_')}",
        },
    )

topics = ["Artificial intelligence", "Machine learning", "Cybersecurity"]

documents = []
for topic in topics:
    doc = fetch_wikipedia_document(topic)
    documents.append(doc)
    print(f"Loaded: {doc.metadata['title']} -> {len(doc.page_content)} characters")

print(f"\nTotal documents loaded: {len(documents)}")
print("\nSample preview:\n", documents[0].page_content[:500])

Loaded: Artificial intelligence -> 85294 characters
Loaded: Machine learning -> 58709 characters
Loaded: Computer security -> 98499 characters

Total documents loaded: 3

Sample preview:
 Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.
High-pr


## Step 4 — Split Documents into Chunks
**Purpose:** LLMs and embedding models work best on small text chunks rather than huge documents. We split each document into overlapping chunks so that relevant passages can be retrieved precisely instead of pulling in an entire article.

**Important info:** `chunk_overlap` keeps some context between chunks so we don't cut a sentence/idea in half at chunk boundaries.

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(documents)
print(f"Total chunks created: {len(chunks)}")
print("\nExample chunk:\n", chunks[0].page_content[:300])


Total chunks created: 504

Example chunk:
 Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develo


## Step 5 — Generate Embeddings & Build the Vector Store
**Purpose:** Convert every text chunk into a numerical vector (embedding) that captures its meaning, then store all vectors inside a FAISS index. This index is what allows fast "semantic search" — finding chunks that are *meaningfully* similar to a user's question, not just keyword matches.

We use the free, local `all-MiniLM-L6-v2` sentence-transformer model — no API key required.

In [5]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(chunks, embedding_model)

# Save the vector store locally so it can be reused (e.g. by the Streamlit app)
vectorstore.save_local("faiss_index")

print("Vector store built and saved to ./faiss_index")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store built and saved to ./faiss_index


## Step 6 — Load the Language Model (LLM)
**Purpose:** This is the model that actually generates natural-language answers, using the retrieved chunks + conversation history as context.

We use `google/flan-t5-base` locally via HuggingFace `transformers` (free, no API key, runs on CPU).

> 🔁 **Want higher quality answers?** Uncomment the OpenAI/Anthropic block below instead (requires an API key) — everything else in the notebook stays unchanged since LangChain abstracts the LLM interface.

In [6]:
!pip install -q langchain-groq

In [13]:
from langchain_groq import ChatGroq
from google.colab import userdata

groq_api_key = userdata.get('GROQ_API_KEY')

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=groq_api_key,
    temperature=0.3,
)

print("LLM ready (using Groq)")

LLM ready (using Groq)


## Step 7 — Set Up Conversation Memory
**Purpose:** `ConversationBufferMemory` stores every past question and answer in the session. On every new turn, LangChain automatically feeds this history back into the chain so the chatbot can resolve references like *"it"*, *"that project"*, or *"what did I just ask?"*.

In [14]:
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)

print("Memory initialized")


Memory initialized


## Step 8 — Build the Conversational RAG Chain
**Purpose:** This is the core of the task — `ConversationalRetrievalChain` wires together:
1. The **retriever** (FAISS vector store → top-k relevant chunks)
2. The **memory** (past conversation turns)
3. The **LLM** (generates the final answer)

Given a new question, it: rewrites the question using chat history (if needed) → retrieves relevant chunks → generates a grounded answer → updates memory.

In [15]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True
)

print("Conversational RAG chain ready")


Conversational RAG chain ready


## Step 9 — Test the Chatbot (Multi-turn, In-Notebook)
**Purpose:** Demonstrate both capabilities together:
- **Retrieval:** answers are grounded in the Wikipedia corpus, not pure model guesswork
- **Memory:** the second question uses "it", relying on the first answer's context to resolve correctly

In [16]:
questions = [
    "What is machine learning?",
    "What are some real-world applications of it?",
    "How is that related to cybersecurity?"
]

for q in questions:
    result = qa_chain.invoke({"question": q})
    print("Q:", q)
    print("A:", result["answer"])
    print("Sources:", [doc.metadata.get("title", "N/A") for doc in result["source_documents"]])
    print("-" * 80)


Q: What is machine learning?
A: Machine learning is the study of programs that can improve their performance on a given task automatically. It is a part of artificial intelligence (AI) that involves the use of statistical methods, fuzzy logic, and probability theory to enable programs to learn and improve without being explicitly programmed.
Sources: ['Artificial intelligence', 'Machine learning', 'Artificial intelligence']
--------------------------------------------------------------------------------
Q: What are some real-world applications of it?
A: The provided context does not explicitly mention specific real-world applications of machine learning. It focuses on the types of machine learning, the history of the field, and the concept of feature learning. I don't know the specific real-world applications of machine learning based on the given context.
Sources: ['Artificial intelligence', 'Machine learning', 'Machine learning']
------------------------------------------------------

## Step 10 — Deploy with Streamlit
**Purpose:** Package the entire pipeline (embeddings → FAISS retriever → memory → LLM chain) into a standalone `app.py` file with a chat UI, so it can run as a real web app with `streamlit run app.py`.

This cell **writes** `app.py` to disk — it rebuilds the vector store from Wikipedia on first run (or you can point it at the saved `faiss_index` folder from Step 5 to skip re-embedding).

In [17]:
%%writefile app.py
import os
import streamlit as st
import requests
from langchain_core.documents import Document
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
try:
    from langchain_classic.memory import ConversationBufferMemory
    from langchain_classic.chains import ConversationalRetrievalChain
except ImportError:
    from langchain.memory import ConversationBufferMemory
    from langchain.chains import ConversationalRetrievalChain
from langchain_groq import ChatGroq

st.set_page_config(page_title="Context-Aware RAG Chatbot", page_icon="🤖")
st.title("🤖 Context-Aware Chatbot (LangChain + RAG)")
st.caption("Ask questions about: Artificial Intelligence, Machine Learning, Cybersecurity")

def fetch_wikipedia_document(title):
    headers = {"User-Agent": "Task4-RAG-Chatbot/1.0 (educational project; contact: student@example.com)"}
    params = {"action": "query", "prop": "extracts", "explaintext": True,
              "titles": title, "format": "json", "redirects": 1}
    resp = requests.get("https://en.wikipedia.org/w/api.php", params=params, headers=headers, timeout=15)
    resp.raise_for_status()
    data = resp.json()
    page = next(iter(data["query"]["pages"].values()))
    content = page.get("extract", "")
    page_title = page.get("title", title)
    return Document(page_content=content,
                     metadata={"title": page_title,
                               "source": f"https://en.wikipedia.org/wiki/{page_title.replace(' ', '_')}"})

@st.cache_resource(show_spinner="Building knowledge base (first run only)...")
def load_chain(groq_api_key):
    topics = ["Artificial intelligence", "Machine learning", "Cybersecurity"]
    documents = [fetch_wikipedia_document(t) for t in topics]

    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    chunks = splitter.split_documents(documents)

    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectorstore = FAISS.from_documents(chunks, embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=groq_api_key, temperature=0.3)

    memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True, output_key="answer")

    chain = ConversationalRetrievalChain.from_llm(
        llm=llm, retriever=retriever, memory=memory, return_source_documents=True
    )
    return chain

# Streamlit app reads the key from an environment variable (set before launch)
groq_api_key = os.environ.get("GROQ_API_KEY", "")
if not groq_api_key:
    st.error("GROQ_API_KEY not found. Please set it before launching the app.")
    st.stop()

qa_chain = load_chain(groq_api_key)

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

user_input = st.chat_input("Ask me something about AI, ML, or Cybersecurity...")

if user_input:
    st.session_state.messages.append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.markdown(user_input)

    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            result = qa_chain.invoke({"question": user_input})
            answer = result["answer"]
            sources = sorted({doc.metadata.get("title", "N/A") for doc in result["source_documents"]})
            st.markdown(answer)
            if sources:
                st.caption("📚 Sources: " + ", ".join(sources))

    st.session_state.messages.append({"role": "assistant", "content": answer})

with st.sidebar:
    st.header("About")
    st.write("This chatbot uses RAG (FAISS + HuggingFace embeddings) to retrieve grounded answers and LangChain's ConversationBufferMemory to remember chat history.")
    if st.button("🗑️ Clear conversation"):
        st.session_state.messages = []
        st.rerun()

Overwriting app.py


### ▶️ How to Run the Streamlit App
1. Make sure `app.py` (written above) is in your working directory.
2. In a terminal, run:
```bash
streamlit run app.py
```
3. Open the local URL shown in the terminal (usually `http://localhost:8501`).
4. Chat with the bot — it retrieves grounded answers from the Wikipedia corpus and remembers the conversation as you go.

> ⚠️ **Important:** Streamlit apps must be run from a terminal/command line, not inside a notebook cell directly. If you're on Google Colab, use a tool like `pyngrok` or `localtunnel` to expose the Streamlit port publicly.


In [18]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧
changed 22 packages in 3s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧

In [19]:
!curl https://loca.lt/mytunnelpassword

34.90.201.103

In [20]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

!pkill -f streamlit
!streamlit run app.py &>/content/logs.txt &

In [21]:
!pip install -q pyngrok

In [22]:
from pyngrok import ngrok
ngrok.set_auth_token("3FwQ9f9OATGblFF8kSiD4acbY9t_2gTVRiqqZE5RsUNtpSSsF")

In [23]:
import time
time.sleep(20)
from pyngrok import ngrok
public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://swab-grader-subatomic.ngrok-free.dev" -> "http://localhost:8501"


## ✅ Conclusion & Results

### What we built
A fully working **context-aware RAG chatbot** that:
1. Loaded a custom knowledge base (Wikipedia articles on AI, ML, and Cybersecurity)
2. Split documents into overlapping chunks for precise retrieval
3. Embedded chunks using `all-MiniLM-L6-v2` and stored them in a FAISS vector index
4. Retrieved the top-3 most relevant chunks for any user question via semantic search
5. Maintained conversational memory across turns using `ConversationBufferMemory`
6. Generated grounded answers with a local LLM (`flan-t5-base`), combining retrieved context + chat history
7. Deployed the entire pipeline as an interactive **Streamlit** chat application

### Results observed
- The chatbot correctly answered direct factual questions (e.g., *"What is machine learning?"*) using retrieved Wikipedia content rather than relying purely on pretrained knowledge.
- Follow-up questions using pronouns (*"it"*, *"that"*) were correctly resolved thanks to conversation memory — demonstrating true multi-turn context awareness.
- Each answer was returned alongside its **source document(s)**, showing the retrieval step was actually grounding the response (a key requirement of RAG, as opposed to plain LLM generation).

### Key takeaways
- **RAG reduces hallucination** by grounding answers in real, retrievable documents instead of only the model's training data.
- **LangChain's chain abstractions** (`ConversationalRetrievalChain`) make it straightforward to combine retrieval + memory + generation without writing this orchestration logic from scratch.
- **The pipeline is modular** — the corpus (Wikipedia → PDFs/docs), the LLM (local flan-t5 → OpenAI/Anthropic), and the vector store (FAISS → Chroma/Pinecone) can all be swapped independently.

### Possible improvements
- Swap `flan-t5-base` for a larger/hosted model (GPT-4o-mini, Claude) for noticeably better answer fluency.
- Add re-ranking of retrieved chunks for higher precision.
- Persist chat history across sessions (currently resets when the Streamlit app restarts).
- Add support for uploading custom PDFs/docs directly from the Streamlit sidebar.
